# HDP — a lossless, portable, governed schema for agent harnesses

One protocol (`hdp.yaml`) that every backend compiles **to** and **from**. This notebook tours the five things that make the repo unique, in five small cells:

1. **The schema** — one 7-layer model (ETCLOVG) for *any* harness.
2. **Lift** — decompile a real backend harness into that schema.
3. **Round-trip identity** — recompile it byte-for-byte (HDP is a *lossless* IR).
4. **Port** — move a harness across backends, with an honest audit of what won't survive.
5. **Guard** — governed edits: propose a change, get it allowed or denied.

> Run from the repo root (where `agents/` lives). `pandas` is used only to render the tables.

## 1. The schema — ETCLOVG in 7 layers

Every harness — NexAU, mini-SWE-agent, OpenHarness — maps onto the same seven layers and the same component vocabulary. That shared model is what makes lift / port / guard backend-agnostic.

In [1]:
import json, os, pandas as pd
assert os.path.exists("agents/code_agent_simple"), "Run this notebook from the repo root."

schema = json.load(open("hdp/schema/hdp.schema.json"))
print("component types:", schema["definitions"]["component"]["properties"]["type"]["enum"])
print("blast-radius ladder:", schema["definitions"]["blastRadius"]["enum"])

gloss = {"execution": "sandbox / isolation", "tooling": "the action space",
         "context": "prompts, memory, skills", "lifecycle": "agent loop + middleware",
         "observability": "tracers", "verification": "reward / verifiers",
         "governance": "blast-radius + evolution policy"}
pd.DataFrame({"layer (E·T·C·L·O·V·G)": list(gloss), "holds": list(gloss.values())})

AssertionError: Run this notebook from the repo root.

## 2. Lift — any harness → the HDP document

`lift` reads a backend harness and returns the typed HDP document. The table below *is* the vendor-neutral schema recovered from the NexAU seed.

In [ ]:
from hdp.engine.lift import lift

doc = lift("agents/code_agent_simple", "/tmp/demo.hdp", target="nexau")  # NexAU harness -> HDP
pd.DataFrame(
    [(layer, c.id, c.type.value, c.blast_radius.value if c.blast_radius else "—")
     for layer, c in doc.components()],
    columns=["layer", "component", "type", "blast_radius"],
)

## 3. Round-trip identity — HDP is a *lossless* IR

`gen(lift(seed))` reproduces the seed harness: every embedded/scaffold file byte-for-byte, the manifest semantically identical, with only a non-runtime attribution sidecar added. This is the validity gate that makes harness *evolution* measurable.

In [ ]:
import shutil
from pathlib import Path
from hdp.engine.core.loader import load
from hdp.engine.gen import generate

seed, regen = Path("agents/code_agent_simple"), Path("/tmp/regen")
if regen.exists():
    shutil.rmtree(regen)
generate(load("/tmp/demo.hdp"), regen, target="nexau")  # HDP -> NexAU harness

files = lambda r: {str(p.relative_to(r)) for p in r.rglob("*") if p.is_file() and "__pycache__" not in p.parts}
byte_identical = [f for f in files(seed) if f != "code_agent.yaml" and (regen / f).read_bytes() == (seed / f).read_bytes()]
print(f"reproduced {len(files(seed))} seed files · added only {files(regen) - files(seed)} · dropped {files(seed) - files(regen) or 'nothing'}")
print(f"{len(byte_identical)} non-manifest files byte-identical ✓")

## 4. Port — cross-backend, with an honest coverage audit

`audit()` enumerates *every* gap at once — `blocking` (won't generate), `silent_drop` (content lands nowhere), `collision` (two components overwrite one file). `port()` then refuses on a blocking gap with that report **instead of crashing**. (Scope: nexau ↔ mini-swe-agent.)

In [ ]:
from hdp.engine.port import audit, port, PortCoverageError

report = audit(doc, "mini-swe-agent", source_target="nexau")  # never writes disk, never raises
print(f"{report.source_target} → {report.dest_target}  |  clean={report.ok}  blocking={len(report.blocking)}")
try:
    port("agents/code_agent_simple", "/tmp/port_out", source_target="nexau", dest_target="mini-swe-agent")
except PortCoverageError as e:
    print("port() refused (enumerable, not a stack trace):", e)

pd.DataFrame(
    [(i.severity, i.component_id, i.reason) for i in report.issues],
    columns=["severity", "component", "reason"],
)

## 5. Guard — governed evolution (the safety brain)

Any proposed edit is evaluated against the governance policy before it lands: a tweak to an *editable* layer is allowed; touching a *read-only* / protected component is denied (and, via `enforce`, rolled back). Same policy, any backend.

In [ ]:
import tempfile
from hdp.engine.core.loader import save
from hdp.engine import guard

EX = "hdp/examples/code-agent-simple.hdp"
old = load(EX)

def edited(mutate):  # copy the doc, apply an edit to its raw YAML, reload
    d = Path(tempfile.mkdtemp()) / "new.hdp"
    shutil.copytree(EX, d)
    nd = load(d); mutate(nd.raw); save(nd)
    return load(d)

find = lambda raw, layer, cid: next(c for c in raw["layers"][layer] if c["id"] == cid)
allow = guard.evaluate(old, edited(lambda r: find(r, "context", "long-term-memory").__setitem__("description", "tweak")),
                       {"changes": [{"component_id": "long-term-memory", "operator": "update"}]})
deny = guard.evaluate(old, edited(lambda r: find(r, "verification", "tb2-verifier").__setitem__("description", "tamper")),
                      {"changes": [{"component_id": "tb2-verifier", "operator": "update"}]})

pd.DataFrame(
    [("edit long-term-memory (context · editable)", allow.ok, "—"),
     ("edit tb2-verifier (verification · read-only)", deny.ok, deny.denied[0].reason if deny.denied else "")],
    columns=["proposed edit", "allowed", "denial reason"],
)